# Full robust ENGIN pipeline — semi-supervised, engine-aware

**Cel:** maksymalizować `Raw_Score = 0.75 * Macro-F1(label) + 0.25 * Accuracy(severity dla uszkodzonych)` bez używania `test.csv` do strojenia.

Kluczowe założenia:
- `train.csv` nie ma etykiet, więc wykorzystujemy `val.csv` jako **labeled reference bank** do pseudo-labelowania train.
- `test.csv` jest ładowany dopiero w ostatniej komórce, wyłącznie do finalnego inference/submission.
- każdy model jest oceniany na silnikach, których nie użyto jako reference.
- severity jest osobnym modelem i dla `ok`/`unknown` zawsze zwracamy `nie_dotyczy`.
- anomaly detection pozostaje niezależnym Isolation Forest.

In [ ]:
from pathlib import Path
import warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupKFold

from src.common import FREQ_COLS, LABELS, FAULT_LABELS
from src.features import FeatureExtractor

SEED = 42
DATA_DIR = Path("data")
RANDOM_STATE = SEED
FAULTS = ["zakoksowany", "lejacy", "pompa", "iglica"]
SEVERITIES = ["male", "srednie", "duze"]

print("READY")

In [ ]:
# 1. Load + strict sanity checks
train_raw = pd.read_csv(DATA_DIR / "train.csv")
val_raw = pd.read_csv(DATA_DIR / "val.csv")

assert "label" not in train_raw.columns, "train.csv unexpectedly contains labels"
assert "label" in val_raw.columns and "severity" in val_raw.columns
assert len(set(train_raw.engine_id) & set(val_raw.engine_id)) == 0

print(f"train rows: {len(train_raw):,} | engines: {train_raw.engine_id.nunique()}")
print(f"val rows:   {len(val_raw):,} | engines: {val_raw.engine_id.nunique()}")
print(f"NaN rate in raw spectra: {train_raw[FREQ_COLS].isna().mean().mean():.2%}")
print("\nValidation labels:")
print(val_raw.label.value_counts(dropna=False))
print("\nValidation severity:")
print(val_raw.severity.value_counts(dropna=False))

In [ ]:
# 2. Robust raw-spectrum imputation.
# Fit ONLY on unlabeled train; this keeps test out of all fitting.
imputer = SimpleImputer(strategy="median")
train_imp = train_raw.copy()
val_imp = val_raw.copy()

train_imp[FREQ_COLS] = imputer.fit_transform(train_raw[FREQ_COLS])
val_imp[FREQ_COLS] = imputer.transform(val_raw[FREQ_COLS])

print("Remaining spectral NaNs:", train_imp[FREQ_COLS].isna().sum().sum(),
      val_imp[FREQ_COLS].isna().sum().sum())

In [ ]:
# 3. Engine-aware feature extraction.
# FeatureExtractor already contains raw spectrum, shape/statistical features,
# spectral peaks/bands and leave-one-cylinder-out engine-relative features.
extractor = FeatureExtractor()
X_train_df = extractor.fit_transform(train_imp)
X_val_df = extractor.transform(val_imp)
feature_names = extractor.get_feature_names()

X_train = np.nan_to_num(X_train_df[feature_names].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val_df[feature_names].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

print(f"Full feature count: {len(feature_names)}")
print("First 15:", feature_names[:15])

In [ ]:
# 4. Feature ranking from a full-feature ExtraTrees model.
def make_prototypes(X_ref, y_ref, classes):
    return {c: np.median(X_ref[y_ref == c], axis=0) for c in classes if np.any(y_ref == c)}

def prototype_pseudo_labels(X, prototypes):
    classes = list(prototypes)
    P = np.vstack([prototypes[c] for c in classes])
    d = np.stack([np.linalg.norm(X-p, axis=1) for p in P], axis=1)
    idx = d.argmin(1)
    order = np.sort(d, axis=1)
    conf = 1.0 - order[:,0] / (order[:,1] + 1e-9)
    conf = np.clip(conf, 0, 1)
    return np.asarray(classes, dtype=object)[idx], conf

def rank_features(X, y, feature_names):
    le = LabelEncoder().fit(y)
    yy = le.transform(y)
    m = ExtraTreesClassifier(
        n_estimators=600, max_features=1.0, max_depth=15,
        min_samples_leaf=2, class_weight="balanced", n_jobs=-1,
        random_state=SEED
    )
    m.fit(X, yy)
    imp = pd.DataFrame({"feature": feature_names, "importance": m.feature_importances_})
    return m, imp.sort_values("importance", ascending=False).reset_index(drop=True)

ref_engines = val_imp.engine_id.drop_duplicates().iloc[:max(1, int(val_imp.engine_id.nunique()*0.6))]
ref_mask = val_imp.engine_id.isin(ref_engines).to_numpy()
ref_labels = val_imp.loc[ref_mask, "label"].astype(str).to_numpy()
classes = [c for c in LABELS if c in set(ref_labels)]
prototypes = make_prototypes(X_val[ref_mask], ref_labels, classes)
pseudo, pseudo_conf = prototype_pseudo_labels(X_train, prototypes)

rank_model, importance = rank_features(X_train, pseudo, feature_names)
TOP_N = 32
selected_features = importance.head(TOP_N).feature.tolist()
selected_idx = [feature_names.index(f) for f in selected_features]

print(f"Selected TOP_N={TOP_N}")
print(importance.head(20).to_string(index=False))

In [ ]:
# 5. Requested simple horizontal feature-importance plot.
top_plot = importance.head(20).sort_values("importance")
plt.figure(figsize=(10, 7))
plt.barh(top_plot.feature, top_plot.importance)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top feature importance — ExtraTrees")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Evaluation helper.
def raw_score(y_true_label, y_pred_label, y_true_sev, y_pred_sev):
    macro = f1_score(y_true_label, y_pred_label, average="macro", zero_division=0)
    fault_mask = np.isin(y_true_label, FAULTS)
    sev_acc = accuracy_score(y_true_sev[fault_mask], y_pred_sev[fault_mask]) if fault_mask.any() else np.nan
    return 0.75 * macro + 0.25 * sev_acc, macro, sev_acc

def build_train_targets(Xtr, Xref, ref_labels, ref_sev, conf_threshold=0.0):
    classes = [c for c in LABELS if c in set(ref_labels)]
    prot = make_prototypes(Xref, ref_labels, classes)
    pseudo, conf = prototype_pseudo_labels(Xtr, prot)
    keep = conf >= conf_threshold
    return pseudo, conf, keep, prot

def fit_label_model(X, y):
    le = LabelEncoder().fit([c for c in LABELS if c in set(y)])
    yy = le.transform(y)
    model = ExtraTreesClassifier(
        n_estimators=150, max_features=1.0, max_depth=15,
        min_samples_leaf=2, class_weight="balanced",
        n_jobs=-1, random_state=SEED
    )
    model.fit(X, yy)
    return model, le

def make_severity_targets(Xtr, pseudo, conf, Xref, ref_labels, ref_sev, threshold):
    X_all, y_all = [], []
    for fault in FAULTS:
        tm = (pseudo == fault) & (conf >= threshold)
        if not tm.any(): continue
        cand = []
        for sev in SEVERITIES:
            rm = (ref_labels == fault) & (ref_sev == sev)
            if rm.any(): cand.append((sev, np.median(Xref[rm], axis=0)))
        if len(cand) < 2: continue
        d = np.stack([np.linalg.norm(Xtr[tm]-p, axis=1) for _,p in cand], axis=1)
        nearest = np.argmin(d, axis=1)
        X_all.append(Xtr[tm]); y_all.extend(np.asarray([c[0] for c in cand], dtype=object)[nearest])
    if not X_all: return np.empty((0, Xtr.shape[1])), np.empty(0, dtype=object)
    return np.vstack(X_all), np.asarray(y_all, dtype=object)

In [ ]:
# 7. Grouped validation on VAL only.
gkf = GroupKFold(n_splits=4)
results = []

for fold, (ref_idx, hold_idx) in enumerate(gkf.split(val_imp, groups=val_imp.engine_id), 1):
    ref_rows = val_imp.iloc[ref_idx]
    hold_rows = val_imp.iloc[hold_idx]
    Xref = X_val[ref_idx]
    Xhold = X_val[hold_idx]
    ref_labels = ref_rows.label.astype(str).to_numpy()
    ref_sev = ref_rows.severity.astype(str).to_numpy()

    for threshold in [0.0, 0.15, 0.25, 0.35]:
        pseudo, conf, keep, _ = build_train_targets(
            X_train[:, selected_idx], Xref[:, selected_idx],
            ref_labels, ref_sev, threshold
        )
        model, le = fit_label_model(X_train[keep][:, selected_idx], pseudo[keep])
        pred_label = le.inverse_transform(model.predict(Xhold[:, selected_idx]))

        sev_X, sev_y = make_severity_targets(
            X_train[:, selected_idx], pseudo, conf, Xref[:, selected_idx],
            ref_labels, ref_sev, threshold
        )
        if len(sev_y) >= 30 and len(np.unique(sev_y)) >= 2:
            sev_le = LabelEncoder().fit(SEVERITIES)
            sev_model = ExtraTreesClassifier(
                n_estimators=250, max_features=0.8, max_depth=8,
                min_samples_leaf=3, class_weight="balanced",
                n_jobs=-1, random_state=SEED
            )
            sev_model.fit(sev_X, sev_le.transform(sev_y))
            pred_sev = np.full(len(hold_rows), "nie_dotyczy", dtype=object)
            fault_hold = np.isin(pred_label, FAULTS)
            pred_sev[fault_hold] = sev_le.inverse_transform(sev_model.predict(Xhold[fault_hold][:, selected_idx]))
        else:
            pred_sev = np.full(len(hold_rows), "nie_dotyczy", dtype=object)

        score, macro, sev_acc = raw_score(
            hold_rows.label.astype(str).to_numpy(), pred_label,
            hold_rows.severity.astype(str).to_numpy(), pred_sev
        )
        results.append({"fold": fold, "threshold": threshold, "raw_score": score,
                        "macro_f1": macro, "severity_accuracy_faults": sev_acc,
                        "pseudo_keep_rate": float(keep.mean())})
        print(f"fold={fold} threshold={threshold:.2f} raw={score:.4f} macro={macro:.4f} sev={sev_acc:.4f} keep={keep.mean():.1%}")

results_df = pd.DataFrame(results)
print("\n=== CV SUMMARY ===")
print(results_df.groupby("threshold")[["raw_score","macro_f1","severity_accuracy_faults","pseudo_keep_rate"]].mean().sort_values("raw_score", ascending=False))

In [ ]:
# 8. Final model using ALL labeled val as reference. TEST IS STILL NOT LOADED.
BEST_THRESHOLD = float(results_df.groupby("threshold")["raw_score"].mean().idxmax())
print("Chosen pseudo-label confidence threshold:", BEST_THRESHOLD)

ref_labels = val_imp.label.astype(str).to_numpy()
ref_sev = val_imp.severity.astype(str).to_numpy()
pseudo, conf, keep, prototypes = build_train_targets(
    X_train[:, selected_idx], X_val[:, selected_idx], ref_labels, ref_sev, BEST_THRESHOLD
)
label_model, label_le = fit_label_model(X_train[keep][:, selected_idx], pseudo[keep])

sev_X, sev_y = make_severity_targets(
    X_train[:, selected_idx], pseudo, conf, X_val[:, selected_idx], ref_labels, ref_sev, BEST_THRESHOLD
)
severity_model = ExtraTreesClassifier(
    n_estimators=250, max_features=0.8, max_depth=8, min_samples_leaf=3,
    class_weight="balanced", n_jobs=-1, random_state=SEED
)
severity_le = LabelEncoder().fit(SEVERITIES)
severity_model.fit(sev_X, severity_le.transform(sev_y))

anomaly_model = IsolationForest(
    n_estimators=300, max_samples="auto", contamination="auto",
    random_state=SEED, n_jobs=-1
)
anomaly_model.fit(X_train[keep][:, selected_idx])

print(f"Final label training rows: {keep.sum():,}/{len(keep):,}")
print(f"Final severity training rows: {len(sev_y):,}")
print("Pseudo-label distribution:")
print(pd.Series(pseudo[keep]).value_counts())

In [ ]:
# 9. Training diagnostic only (NOT a validation metric).
train_pred = label_le.inverse_transform(label_model.predict(X_train[keep][:, selected_idx]))
print("Training pseudo-label accuracy (NOT a validation metric):", accuracy_score(pseudo[keep], train_pred))

final_imp = pd.DataFrame({"feature": selected_features, "importance": label_model.feature_importances_}).sort_values("importance", ascending=False)
print("\nFinal model feature importance:")
print(final_imp.head(20).to_string(index=False))

plt.figure(figsize=(10, 7))
p = final_imp.head(20).sort_values("importance")
plt.barh(p.feature, p.importance)
plt.xlabel("Importance")
plt.title("Final ExtraTrees label model — top features")
plt.tight_layout()
plt.show()

In [ ]:
# 10. ONLY NOW load TEST for final inference.
test_raw = pd.read_csv(DATA_DIR / "test.csv")
assert "label" not in test_raw.columns and "severity" not in test_raw.columns

test_imp = test_raw.copy()
test_imp[FREQ_COLS] = imputer.transform(test_raw[FREQ_COLS])
test_feat_df = extractor.transform(test_imp)
X_test = np.nan_to_num(test_feat_df[feature_names].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0)[:, selected_idx]

pred_label = label_le.inverse_transform(label_model.predict(X_test))
pred_severity = np.full(len(test_raw), "nie_dotyczy", dtype=object)
fault_mask = np.isin(pred_label, FAULTS)
pred_severity[fault_mask] = severity_le.inverse_transform(severity_model.predict(X_test[fault_mask]))

anomaly_raw = anomaly_model.decision_function(X_test)
anomaly_score = np.clip(0.5 - anomaly_raw, 0, 1)

predictions = pd.DataFrame({
    "engine_id": test_raw.engine_id,
    "cylinder": test_raw.cylinder,
    "label": pred_label,
    "severity": pred_severity,
    "anomaly_score": anomaly_score
})
predictions.to_csv("predictions.csv", index=False)

print("\n=== FINAL TEST PREDICTIONS ===")
print(predictions.head(20).to_string(index=False))
print("\nLabel distribution:")
print(predictions.label.value_counts())
print("\nSeverity distribution:")
print(predictions.severity.value_counts())
print(f"Saved: predictions.csv ({len(predictions):,} rows)")

## Important interpretation

The final `predictions.csv` is the **only artifact produced from `test.csv`**, and test labels are never read.

Before submission, use the grouped CV table to decide whether the final threshold / feature count should be changed. Do not optimize against test predictions.

Competition score: `0.75 * Macro-F1(label) + 0.25 * Accuracy(severity for faulty cylinders)`.